In [1]:
import pandas as pd
import numpy as np

telemetry = pd.read_csv(
    "Preprocessed_Data/telemetry_clean.csv"
)

trips = pd.read_csv(
    "Preprocessed_Data/trips_clean.csv"
)

drivers = pd.read_csv(
    "Preprocessed_Data/drivers_clean.csv"
)


In [2]:
telemetry["Timestamp"] = pd.to_datetime(
    telemetry["Timestamp"],
    errors="coerce"
)

In [3]:
telemetry = telemetry.sort_values(
    ["Trip_ID", "Timestamp"]
).reset_index(drop=True)


In [4]:
telemetry["Accel_Magnitude_g"] = np.sqrt(
    telemetry["Accel_X_g"]**2 +
    telemetry["Accel_Y_g"]**2 +
    telemetry["Accel_Z_g"]**2
)


In [5]:
telemetry[
    [
        "Accel_X_g",
        "Accel_Y_g",
        "Accel_Z_g",
        "Accel_Magnitude_g"
    ]
].head()


,Accel_X_g,Accel_Y_g,Accel_Z_g,Accel_Magnitude_g
0,0.070,-0.035,0.998,1.001064
1,0.035,0.013,0.968,0.968720
2,-0.096,0.029,1.049,1.053783
3,-0.505,0.020,1.027,1.144620
4,0.017,0.046,0.982,0.983224


In [6]:
telemetry["Accel_Change_g"] = (
    telemetry
    .groupby("Trip_ID")["Accel_Magnitude_g"]
    .diff()
)

In [7]:
telemetry["Abs_Accel_Change_g"] = (
    telemetry["Accel_Change_g"].abs()
)


In [8]:
telemetry[
    [
        "Trip_ID",
        "Timestamp",
        "Accel_Magnitude_g",
        "Accel_Change_g",
        "Abs_Accel_Change_g"
    ]
].head(15)

,Trip_ID,Timestamp,Accel_Magnitude_g,Accel_Change_g,Abs_Accel_Change_g
0,T00001,2026-08-04 11:08:00,1.001064,NaN,NaN
1,T00001,2026-08-04 11:09:00,0.968720,-0.032344,0.032344
2,T00001,2026-08-04 11:10:00,1.053783,0.085063,0.085063
3,T00001,2026-08-04 11:11:00,1.144620,0.090837,0.090837
4,T00001,2026-08-04 11:12:00,0.983224,-0.161396,0.161396
5,T00001,2026-08-04 11:13:00,0.965411,-0.017813,0.017813
6,T00001,2026-08-04 11:14:00,1.059468,0.094057,0.094057
7,T00001,2026-08-04 11:15:00,0.942719,-0.116749,0.116749
8,T00001,2026-08-04 11:16:00,1.000688,0.057968,0.057968
9,T00001,2026-08-04 11:17:00,1.044570,0.043882,0.043882


In [9]:
telemetry["Abs_Accel_Change_g"].describe()

count    12537.000000
mean         0.071434
std          0.096925
min          0.000006
25%          0.020753
50%          0.046083
75%          0.082910
max          0.959304
Name: Abs_Accel_Change_g, dtype: float64

In [10]:
telemetry["Abs_Accel_Change_g"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99]
)

0.50    0.046083
0.75    0.082910
0.90    0.145988
0.95    0.216216
0.99    0.587690
Name: Abs_Accel_Change_g, dtype: float64

In [11]:

harsh_accel_threshold = telemetry[
    "Abs_Accel_Change_g"
].quantile(0.95)

print(harsh_accel_threshold)


0.21621628404960402


In [12]:
telemetry["Speed_Change_kmph"] = (
    telemetry
    .groupby("Trip_ID")["Speed_kmph"]
    .diff()
)

In [13]:
telemetry["Abs_Speed_Change_kmph"] = (
    telemetry["Speed_Change_kmph"].abs()
)

In [14]:
telemetry[
    [
        "Trip_ID",
        "Timestamp",
        "Speed_kmph",
        "Speed_Change_kmph"
    ]
].head(20)

,Trip_ID,Timestamp,Speed_kmph,Speed_Change_kmph
0,T00001,2026-08-04 11:08:00,22.1,NaN
1,T00001,2026-08-04 11:09:00,37.9,15.8
2,T00001,2026-08-04 11:10:00,20.0,-17.9
3,T00001,2026-08-04 11:11:00,27.9,7.9
4,T00001,2026-08-04 11:12:00,23.6,-4.3
5,T00001,2026-08-04 11:13:00,36.6,13.0
6,T00001,2026-08-04 11:14:00,39.4,2.8
7,T00001,2026-08-04 11:15:00,23.5,-15.9
8,T00001,2026-08-04 11:16:00,22.7,-0.8
9,T00001,2026-08-04 11:17:00,30.0,7.3


In [15]:
telemetry["Speed_Change_kmph"].describe()

count    12537.000000
mean         0.008646
std         14.781052
min        -70.500000
25%         -7.900000
50%          0.100000
75%          8.000000
max         63.500000
Name: Speed_Change_kmph, dtype: float64

In [16]:
telemetry["Speed_Change_kmph"].quantile(
    [0.90, 0.95, 0.99]
)


0.90    18.500
0.95    25.400
0.99    36.964
Name: Speed_Change_kmph, dtype: float64

In [17]:
accel_speed_threshold = telemetry[
    "Speed_Change_kmph"
].quantile(0.95)


In [18]:
telemetry["Harsh_Acceleration"] = (
    telemetry["Speed_Change_kmph"]
    >= accel_speed_threshold
)

In [19]:
telemetry["Harsh_Acceleration"].value_counts()

Harsh_Acceleration
False    12359
True       628
Name: count, dtype: int64

In [20]:
brake_speed_threshold = telemetry[
    "Speed_Change_kmph"
].quantile(0.05)

In [21]:
print(brake_speed_threshold)

-25.5


In [23]:
telemetry["Harsh_Braking"] = (
    telemetry["Speed_Change_kmph"]
    <= brake_speed_threshold
)


In [24]:
telemetry["Harsh_Braking"].value_counts()


Harsh_Braking
False    12359
True       628
Name: count, dtype: int64

In [25]:
telemetry["Gyro_Magnitude_dps"] = np.sqrt(
    telemetry["Gyro_X_dps"]**2 +
    telemetry["Gyro_Y_dps"]**2 +
    telemetry["Gyro_Z_dps"]**2
)


In [26]:
telemetry[
    [
        "Gyro_X_dps",
        "Gyro_Y_dps",
        "Gyro_Z_dps",
        "Gyro_Magnitude_dps"
    ]
].head()

,Gyro_X_dps,Gyro_Y_dps,Gyro_Z_dps,Gyro_Magnitude_dps
0,1.49,1.09,-1.33,2.275324
1,-0.64,-0.94,-1.28,1.712192
2,-0.70,-0.92,1.72,2.072390
3,-0.62,0.91,-1.32,1.718982
4,-0.16,-3.37,-2.89,4.442364


In [27]:

telemetry["Gyro_Change_dps"] = (
    telemetry
    .groupby("Trip_ID")["Gyro_Magnitude_dps"]
    .diff()
)

In [28]:
telemetry["Abs_Gyro_Change_dps"] = (
    telemetry["Gyro_Change_dps"].abs()
)


In [29]:
telemetry["Abs_Gyro_Change_dps"].describe()

count    12537.000000
mean         4.364849
std          9.810413
min          0.000215
25%          0.660697
50%          1.416527
75%          2.592874
max         55.913359
Name: Abs_Gyro_Change_dps, dtype: float64

In [30]:
telemetry["Abs_Gyro_Change_dps"].quantile(
    [0.90, 0.95, 0.99]
)

0.90     4.971397
0.95    32.989030
0.99    47.530901
Name: Abs_Gyro_Change_dps, dtype: float64

In [31]:
turn_threshold = telemetry[
    "Abs_Gyro_Change_dps"
].quantile(0.95)


In [32]:
telemetry["Sudden_Turn"] = (
    telemetry["Abs_Gyro_Change_dps"]
    >= turn_threshold
)

In [33]:
speed_threshold = telemetry[
    "Speed_kmph"
].quantile(0.95)

print(speed_threshold)


41.8


In [34]:
telemetry["High_Speed_Event"] = (
    telemetry["Speed_kmph"] >= speed_threshold
)

In [35]:
event_columns = [
    "Harsh_Acceleration",
    "Harsh_Braking",
    "Sudden_Turn",
    "High_Speed_Event"
]

for col in event_columns:
    telemetry[col] = telemetry[col].astype(int)




In [36]:
trip_behaviour = telemetry.groupby(
    "Trip_ID"
).agg(
    Harsh_Acceleration_Count=(
        "Harsh_Acceleration",
        "sum"
    ),
    Harsh_Braking_Count=(
        "Harsh_Braking",
        "sum"
    ),
    Sudden_Turn_Count=(
        "Sudden_Turn",
        "sum"
    ),
    High_Speed_Count=(
        "High_Speed_Event",
        "sum"
    ),
    Avg_Speed=("Speed_kmph", "mean"),
    Max_Speed=("Speed_kmph", "max"),
    Avg_Acceleration=("Accel_Magnitude_g", "mean"),
    Max_Acceleration=("Accel_Magnitude_g", "max"),
    Avg_Gyro=("Gyro_Magnitude_dps", "mean"),
    Max_Gyro=("Gyro_Magnitude_dps", "max")
).reset_index()

In [37]:
display(trip_behaviour.head())

,Trip_ID,Harsh_Acceleration_Count,Harsh_Braking_Count,Sudden_Turn_Count,High_Speed_Count,Avg_Speed,Max_Speed,Avg_Acceleration,Max_Acceleration,Avg_Gyro,Max_Gyro
0,T00001,0,0,0,0,28.718750,39.4,1.015943,1.186011,2.626879,5.054196
1,T00002,1,2,2,3,28.545455,54.4,0.998121,1.070320,4.626562,36.400853
2,T00003,0,0,2,0,16.900000,32.7,1.016800,1.191220,4.699843,38.313713
3,T00004,0,0,3,0,23.907143,40.4,1.004191,1.223474,7.708968,54.095156
4,T00005,0,1,1,2,28.707692,42.5,1.040415,1.265865,6.098402,46.657554


In [38]:
trip_behaviour = trip_behaviour.merge(
    trips[
        [
            "Trip_ID",
            "Driver_ID",
            "Vehicle_ID",
            "Trip_Date",
            "Duration_Min",
            "Distance_KM",
            "Avg_Speed_kmph",
            "Max_Speed_kmph"
        ]
    ],
    on="Trip_ID",
    how="left"
)

In [39]:
trip_behaviour["Risk_Events"] = (
    trip_behaviour["Harsh_Acceleration_Count"] +
    trip_behaviour["Harsh_Braking_Count"] +
    trip_behaviour["Sudden_Turn_Count"] +
    trip_behaviour["High_Speed_Count"]
)

In [40]:
trip_behaviour["Risk_Events_Per_10_Min"] = (
    trip_behaviour["Risk_Events"] /
    trip_behaviour["Duration_Min"] * 10
)


In [41]:
driver_behaviour = trip_behaviour.groupby(
    "Driver_ID"
).agg(
    Total_Trips=("Trip_ID", "nunique"),

    Total_Harsh_Acceleration=(
        "Harsh_Acceleration_Count",
        "sum"
    ),

    Total_Harsh_Braking=(
        "Harsh_Braking_Count",
        "sum"
    ),

    Total_Sudden_Turns=(
        "Sudden_Turn_Count",
        "sum"
    ),

    Total_High_Speed_Events=(
        "High_Speed_Count",
        "sum"
    ),

    Avg_Risk_Events_Per_10_Min=(
        "Risk_Events_Per_10_Min",
        "mean"
    ),

    Avg_Speed=("Avg_Speed", "mean"),

    Max_Speed=("Max_Speed", "max"),

    Avg_Acceleration=(
        "Avg_Acceleration",
        "mean"
    ),

    Max_Acceleration=(
        "Max_Acceleration",
        "max"
    ),

    Avg_Gyro=("Avg_Gyro", "mean"),

    Max_Gyro=("Max_Gyro", "max")
).reset_index()

In [42]:
display(driver_behaviour)

,Driver_ID,Total_Trips,Total_Harsh_Acceleration,Total_Harsh_Braking,Total_Sudden_Turns,Total_High_Speed_Events,Avg_Risk_Events_Per_10_Min,Avg_Speed,Max_Speed,Avg_Acceleration,Max_Acceleration,Avg_Gyro,Max_Gyro
0,D01,15,11,14,31,19,1.789044,25.381572,54.4,1.026368,1.639733,5.231414,54.095156
1,D02,15,15,20,18,13,1.837993,22.676931,50.9,1.049962,1.813246,5.097916,52.919581
2,D03,15,44,44,30,45,3.370873,22.852751,68.8,1.033505,1.476075,5.736858,54.574108
3,D04,15,13,15,27,22,1.634954,24.292656,51.6,1.013047,1.561136,5.287398,54.671394
4,D05,15,10,7,12,2,0.565998,23.017252,47.8,1.011880,1.662806,3.934284,45.815206
5,D06,15,57,48,35,73,4.461658,25.581106,73.6,1.022332,1.372765,5.539019,54.201787
6,D07,15,14,19,26,24,1.870861,25.276740,54.4,1.013736,1.493077,4.671597,53.351924
7,D08,15,16,14,22,16,1.402142,24.610112,56.3,1.016041,1.783398,4.327752,52.290292
8,D09,15,13,9,3,0,0.583109,25.708229,40.7,1.011136,1.760416,3.462914,52.667603
9,D10,15,13,14,24,13,1.310201,21.464280,56.7,1.025164,1.788168,4.579929,55.000745


In [43]:
driver_behaviour = driver_behaviour.merge(
    drivers[
        [
            "Driver_ID",
            "Driver_Name",
            "Age",
            "Gender",
            "License_Experience_Years",
            "Home_Hub"
        ]
    ],
    on="Driver_ID",
    how="left"
)


In [44]:
driver_behaviour.to_csv(
    "Preprocessed_Data/driver_behaviour_features.csv",
    index=False
)

In [45]:

trip_behaviour.to_csv(
    "Preprocessed_Data/trip_behaviour_features.csv",
    index=False
)

In [46]:
print("Speed change percentiles:")
print(
    telemetry["Speed_Change_kmph"].quantile(
        [0.01, 0.05, 0.50, 0.95, 0.99]
    )
)

print("\nAcceleration change percentiles:")
print(
    telemetry["Abs_Accel_Change_g"].quantile(
        [0.50, 0.90, 0.95, 0.99]
    )
)

print("\nGyroscope change percentiles:")
print(
    telemetry["Abs_Gyro_Change_dps"].quantile(
        [0.50, 0.90, 0.95, 0.99]
    )
)

print("\nSpeed percentiles:")
print(
    telemetry["Speed_kmph"].quantile(
        [0.50, 0.90, 0.95, 0.99]
    )
)


Speed change percentiles:
0.01   -37.764
0.05   -25.500
0.50     0.100
0.95    25.400
0.99    36.964
Name: Speed_Change_kmph, dtype: float64

Acceleration change percentiles:
0.50    0.046083
0.90    0.145988
0.95    0.216216
0.99    0.587690
Name: Abs_Accel_Change_g, dtype: float64

Gyroscope change percentiles:
0.50     1.416527
0.90     4.971397
0.95    32.989030
0.99    47.530901
Name: Abs_Gyro_Change_dps, dtype: float64

Speed percentiles:
0.50    24.9
0.90    37.4
0.95    41.8
0.99    51.3
Name: Speed_kmph, dtype: float64


In [47]:
import pandas as pd
import numpy as np

# Load cleaned datasets
telemetry = pd.read_csv(
    "Preprocessed_Data/telemetry_clean.csv"
)

trips = pd.read_csv(
    "Preprocessed_Data/trips_clean.csv"
)

vehicles = pd.read_csv(
    "Preprocessed_Data/vehicles_clean.csv"
)

# Convert timestamp
telemetry["Timestamp"] = pd.to_datetime(
    telemetry["Timestamp"],
    errors="coerce"
)

# Sort telemetry chronologically within each trip
telemetry = telemetry.sort_values(
    ["Trip_ID", "Timestamp"]
).reset_index(drop=True)

print("Telemetry shape:", telemetry.shape)
print("Trips shape:", trips.shape)
print("Vehicles shape:", vehicles.shape)


Telemetry shape: (12987, 13)
Trips shape: (450, 16)
Vehicles shape: (30, 8)


In [48]:
telemetry["Accel_Magnitude_g"] = np.sqrt(
    telemetry["Accel_X_g"]**2 +
    telemetry["Accel_Y_g"]**2 +
    telemetry["Accel_Z_g"]**2
)

In [49]:
print(
    telemetry[
        [
            "Accel_X_g",
            "Accel_Y_g",
            "Accel_Z_g",
            "Accel_Magnitude_g"
        ]
    ].head()
)


   Accel_X_g  Accel_Y_g  Accel_Z_g  Accel_Magnitude_g
0      0.070     -0.035      0.998           1.001064
1      0.035      0.013      0.968           0.968720
2     -0.096      0.029      1.049           1.053783
3     -0.505      0.020      1.027           1.144620
4      0.017      0.046      0.982           0.983224


In [50]:
telemetry["Gyro_Magnitude_dps"] = np.sqrt(
    telemetry["Gyro_X_dps"]**2 +
    telemetry["Gyro_Y_dps"]**2 +
    telemetry["Gyro_Z_dps"]**2
)

In [51]:
accel_trip = telemetry.groupby(
    "Trip_ID"
).agg(
    Accel_Mean=("Accel_Magnitude_g", "mean"),
    Accel_Std=("Accel_Magnitude_g", "std"),
    Accel_Max=("Accel_Magnitude_g", "max"),
    Accel_Min=("Accel_Magnitude_g", "min")
).reset_index()

In [52]:
accel_trip["Accel_Range"] = (
    accel_trip["Accel_Max"] -
    accel_trip["Accel_Min"]
)

In [53]:
display(accel_trip.head())

,Trip_ID,Accel_Mean,Accel_Std,Accel_Max,Accel_Min,Accel_Range
0,T00001,1.015943,0.071932,1.186011,0.905855,0.280155
1,T00002,0.998121,0.043172,1.070320,0.895457,0.174863
2,T00003,1.016800,0.084219,1.191220,0.881292,0.309928
3,T00004,1.004191,0.059430,1.223474,0.871294,0.352180
4,T00005,1.040415,0.091496,1.265865,0.904279,0.361585


In [54]:

gyro_trip = telemetry.groupby(
    "Trip_ID"
).agg(
    Gyro_Mean=("Gyro_Magnitude_dps", "mean"),
    Gyro_Std=("Gyro_Magnitude_dps", "std"),
    Gyro_Max=("Gyro_Magnitude_dps", "max"),
    Gyro_Min=("Gyro_Magnitude_dps", "min")
).reset_index()

In [55]:
gyro_trip["Gyro_Range"] = (
    gyro_trip["Gyro_Max"] -
    gyro_trip["Gyro_Min"]
)


In [56]:
accel_spike_threshold = telemetry[
    "Accel_Magnitude_g"
].quantile(0.95)

print(
    "Acceleration spike threshold:",
    accel_spike_threshold
)

Acceleration spike threshold: 1.1410279134176757


In [57]:
telemetry["Accel_Spike"] = (
    telemetry["Accel_Magnitude_g"]
    >= accel_spike_threshold
).astype(int)

In [58]:
gyro_spike_threshold = telemetry[
    "Gyro_Magnitude_dps"
].quantile(0.95)

print(
    "Gyroscope magnitude spike threshold:",
    gyro_spike_threshold
)

Gyroscope magnitude spike threshold: 7.575581189337716


In [59]:
telemetry["Gyro_Spike"] = (
    telemetry["Gyro_Magnitude_dps"]
    >= gyro_spike_threshold
).astype(int)


In [60]:
spikes_trip = telemetry.groupby(
    "Trip_ID"
).agg(
    Accel_Spike_Count=("Accel_Spike", "sum"),
    Gyro_Spike_Count=("Gyro_Spike", "sum")
).reset_index()


In [61]:
spikes_trip["Total_Sensor_Spikes"] = (
    spikes_trip["Accel_Spike_Count"] +
    spikes_trip["Gyro_Spike_Count"]
)


In [62]:
display(spikes_trip.head())

,Trip_ID,Accel_Spike_Count,Gyro_Spike_Count,Total_Sensor_Spikes
0,T00001,2,0,2
1,T00002,0,1,1
2,T00003,4,1,5
3,T00004,1,5,6
4,T00005,1,1,2


In [63]:
spikes_trip = spikes_trip.merge(
    trips[
        [
            "Trip_ID",
            "Duration_Min"
        ]
    ],
    on="Trip_ID",
    how="left"
)

In [64]:
spikes_trip["Accel_Spikes_Per_10_Min"] = (
    spikes_trip["Accel_Spike_Count"] /
    spikes_trip["Duration_Min"] * 10
)

In [65]:
spikes_trip["Gyro_Spikes_Per_10_Min"] = (
    spikes_trip["Gyro_Spike_Count"] /
    spikes_trip["Duration_Min"] * 10
)

In [66]:
spikes_trip["Total_Spikes_Per_10_Min"] = (
    spikes_trip["Total_Sensor_Spikes"] /
    spikes_trip["Duration_Min"] * 10
)

In [67]:
vehicle_trip_features = accel_trip.merge(
    gyro_trip,
    on="Trip_ID",
    how="left"
)


In [68]:
vehicle_trip_features = vehicle_trip_features.merge(
    spikes_trip,
    on="Trip_ID",
    how="left"
)


In [69]:
vehicle_trip_features = vehicle_trip_features.merge(
    trips[
        [
            "Trip_ID",
            "Driver_ID",
            "Vehicle_ID",
            "Trip_Date",
            "Duration_Min",
            "Distance_KM",
            "Avg_Speed_kmph",
            "Max_Speed_kmph"
        ]
    ],
    on="Trip_ID",
    how="left"
)


In [70]:
print(
    "Vehicle-trip feature shape:",
    vehicle_trip_features.shape
)

display(vehicle_trip_features.head())


Vehicle-trip feature shape: (450, 25)


,Trip_ID,Accel_Mean,Accel_Std,Accel_Max,Accel_Min,Accel_Range,Gyro_Mean,Gyro_Std,Gyro_Max,Gyro_Min,...,Accel_Spikes_Per_10_Min,Gyro_Spikes_Per_10_Min,Total_Spikes_Per_10_Min,Driver_ID,Vehicle_ID,Trip_Date,Duration_Min_y,Distance_KM,Avg_Speed_kmph,Max_Speed_kmph
0,T00001,1.015943,0.071932,1.186011,0.905855,0.280155,2.626879,1.067270,5.054196,0.856213,...,1.250000,0.000000,1.250000,D01,V01,2026-08-04,16,9.34,28.7,39.4
1,T00002,0.998121,0.043172,1.070320,0.895457,0.174863,4.626562,7.208374,36.400853,0.831565,...,0.000000,0.454545,0.454545,D01,V01,2026-08-02,22,5.00,28.5,54.4
2,T00003,1.016800,0.084219,1.191220,0.881292,0.309928,4.699843,6.789118,38.313713,2.043257,...,1.481481,0.370370,1.851852,D01,V01,2026-07-31,27,7.25,16.9,32.7
3,T00004,1.004191,0.059430,1.223474,0.871294,0.352180,7.708968,12.728351,54.095156,1.059670,...,0.238095,1.190476,1.428571,D01,V01,2026-07-31,42,9.27,23.9,40.4
4,T00005,1.040415,0.091496,1.265865,0.904279,0.361585,6.098402,12.215527,46.657554,1.755506,...,0.769231,0.769231,1.538462,D01,V01,2026-08-01,13,7.74,28.7,42.5


In [72]:
from sklearn.preprocessing import StandardScaler



scaler = StandardScaler()

vehicle_trip_features[
    [
        "Accel_Variability_Z",
        "Gyro_Variability_Z"
    ]
] = scaler.fit_transform(
    vehicle_trip_features[
        [
            "Accel_Std",
            "Gyro_Std"
        ]
    ].fillna(0)
)


In [73]:
vehicle_trip_features[
    "Sensor_Variability_Index"
] = (
    vehicle_trip_features["Accel_Variability_Z"] +
    vehicle_trip_features["Gyro_Variability_Z"]
) / 2

In [74]:
vehicle_health = vehicle_trip_features.groupby(
    "Vehicle_ID"
).agg(
    Total_Trips=("Trip_ID", "nunique"),

    Avg_Accel_Variability=(
        "Accel_Std",
        "mean"
    ),

    Max_Accel_Variability=(
        "Accel_Std",
        "max"
    ),

    Avg_Accel_Range=(
        "Accel_Range",
        "mean"
    ),

    Avg_Gyro_Variability=(
        "Gyro_Std",
        "mean"
    ),

    Max_Gyro_Variability=(
        "Gyro_Std",
        "max"
    ),

    Avg_Gyro_Range=(
        "Gyro_Range",
        "mean"
    ),

    Avg_Accel_Spikes_Per_10_Min=(
        "Accel_Spikes_Per_10_Min",
        "mean"
    ),

    Avg_Gyro_Spikes_Per_10_Min=(
        "Gyro_Spikes_Per_10_Min",
        "mean"
    ),

    Avg_Total_Sensor_Spikes=(
        "Total_Spikes_Per_10_Min",
        "mean"
    ),

    Avg_Sensor_Variability=(
        "Sensor_Variability_Index",
        "mean"
    )
).reset_index()

In [75]:
print("Vehicles:", vehicle_health.shape)

display(vehicle_health.head())


Vehicles: (30, 12)


,Vehicle_ID,Total_Trips,Avg_Accel_Variability,Max_Accel_Variability,Avg_Accel_Range,Avg_Gyro_Variability,Max_Gyro_Variability,Avg_Gyro_Range,Avg_Accel_Spikes_Per_10_Min,Avg_Gyro_Spikes_Per_10_Min,Avg_Total_Sensor_Spikes,Avg_Sensor_Variability
0,V01,15,0.081490,0.180949,0.376312,8.136740,12.981530,38.011426,0.734773,0.591845,1.326617,0.278180
1,V02,16,0.150929,0.255354,0.663612,6.074134,13.277175,27.148729,1.156757,0.890994,2.047751,0.808013
2,V03,16,0.066163,0.105605,0.291966,8.524780,13.232653,40.321897,0.917712,0.694132,1.611844,0.153029
3,V04,15,0.051336,0.130241,0.248070,7.761656,11.435348,35.766309,0.246294,0.627415,0.873708,-0.106867
4,V05,15,0.061867,0.143864,0.304280,4.584335,8.466150,25.626137,0.176754,0.294238,0.470992,-0.375826


In [76]:
vehicle_health = vehicle_health.merge(
    vehicles[
        [
            "Vehicle_ID",
            "Vehicle_Type",
            "Make",
            "Model",
            "Manufacture_Year",
            "Registration_Date",
            "Odometer_KM_Start_of_Week",
            "Last_Service_Date"
        ]
    ],
    on="Vehicle_ID",
    how="left"
)

In [77]:

vehicle_health["Vehicle_Age_Years"] = (
    2026 -
    vehicle_health["Manufacture_Year"]
)

In [78]:
display(
    vehicle_health[
        [
            "Vehicle_ID",
            "Manufacture_Year",
            "Vehicle_Age_Years"
        ]
    ].head()
)

,Vehicle_ID,Manufacture_Year,Vehicle_Age_Years
0,V01,2021,5
1,V02,2021,5
2,V03,2022,4
3,V04,2020,6
4,V05,2025,1


In [79]:
vehicle_health["Last_Service_Date"] = pd.to_datetime(
    vehicle_health["Last_Service_Date"],
    errors="coerce"
)


In [80]:
vehicle_health["Days_Since_Last_Service"] = (
    pd.Timestamp("2026-08-21") -
    vehicle_health["Last_Service_Date"]
).dt.days


In [81]:
display(
    vehicle_health[
        [
            "Vehicle_ID",
            "Last_Service_Date",
            "Days_Since_Last_Service"
        ]
    ].head()
)

,Vehicle_ID,Last_Service_Date,Days_Since_Last_Service
0,V01,2026-06-08,74
1,V02,2026-05-06,107
2,V03,2026-07-08,44
3,V04,2026-07-19,33
4,V05,2026-07-12,40


In [82]:
vehicle_health[
    "Accel_Variability_Percentile"
] = (
    vehicle_health[
        "Avg_Accel_Variability"
    ].rank(pct=True)
)

In [83]:



vehicle_health[
    "Gyro_Variability_Percentile"
] = (
    vehicle_health[
        "Avg_Gyro_Variability"
    ].rank(pct=True)
)


vehicle_health[
    "Sensor_Spike_Percentile"
] = (
    vehicle_health[
        "Avg_Total_Sensor_Spikes"
    ].rank(pct=True)
)


In [84]:
display(
    vehicle_health[
        [
            "Vehicle_ID",
            "Accel_Variability_Percentile",
            "Gyro_Variability_Percentile",
            "Sensor_Spike_Percentile"
        ]
    ].sort_values(
        "Accel_Variability_Percentile",
        ascending=False
    )
)


,Vehicle_ID,Accel_Variability_Percentile,Gyro_Variability_Percentile,Sensor_Spike_Percentile
1,V02,1.000000,0.433333,1.000000
18,V19,0.966667,0.966667,0.966667
24,V25,0.933333,0.633333,0.766667
22,V23,0.900000,0.833333,0.900000
12,V13,0.866667,0.600000,0.633333
14,V15,0.833333,0.766667,0.600000
23,V24,0.800000,1.000000,0.800000
13,V14,0.766667,0.733333,0.933333
11,V12,0.733333,0.800000,0.833333
0,V01,0.700000,0.700000,0.733333


In [85]:
vehicle_health[
    "High_Accel_Variability"
] = (
    vehicle_health[
        "Accel_Variability_Percentile"
    ] >= 0.95
).astype(int)



vehicle_health[
    "High_Gyro_Variability"
] = (
    vehicle_health[
        "Gyro_Variability_Percentile"
    ] >= 0.95
).astype(int)



vehicle_health[
    "High_Sensor_Spikes"
] = (
    vehicle_health[
        "Sensor_Spike_Percentile"
    ] >= 0.95
).astype(int)

In [86]:
vehicle_health[
    "Anomaly_Indicator_Count"
] = (
    vehicle_health["High_Accel_Variability"] +
    vehicle_health["High_Gyro_Variability"] +
    vehicle_health["High_Sensor_Spikes"]
)


In [87]:
display(
    vehicle_health[
        [
            "Vehicle_ID",
            "High_Accel_Variability",
            "High_Gyro_Variability",
            "High_Sensor_Spikes",
            "Anomaly_Indicator_Count"
        ]
    ].sort_values(
        "Anomaly_Indicator_Count",
        ascending=False
    )
)


,Vehicle_ID,High_Accel_Variability,High_Gyro_Variability,High_Sensor_Spikes,Anomaly_Indicator_Count
18,V19,1,1,1,3
1,V02,1,0,1,2
23,V24,0,1,0,1
3,V04,0,0,0,0
2,V03,0,0,0,0
0,V01,0,0,0,0
6,V07,0,0,0,0
7,V08,0,0,0,0
8,V09,0,0,0,0
9,V10,0,0,0,0


In [88]:
vehicle_health[
    "Maintenance_Candidate"
] = (
    vehicle_health[
        "Anomaly_Indicator_Count"
    ] >= 2
).astype(int)

In [89]:
print(
    vehicle_health[
        "Maintenance_Candidate"
    ].value_counts()
)


Maintenance_Candidate
0    28
1     2
Name: count, dtype: int64


In [90]:
vehicle_health_sorted = vehicle_health.sort_values(
    [
        "Anomaly_Indicator_Count",
        "Avg_Sensor_Variability"
    ],
    ascending=False
)

In [91]:
display(
    vehicle_health_sorted[
        [
            "Vehicle_ID",
            "Make",
            "Model",
            "Avg_Accel_Variability",
            "Avg_Gyro_Variability",
            "Avg_Total_Sensor_Spikes",
            "Anomaly_Indicator_Count",
            "Maintenance_Candidate"
        ]
    ]
)

,Vehicle_ID,Make,Model,Avg_Accel_Variability,Avg_Gyro_Variability,Avg_Total_Sensor_Spikes,Anomaly_Indicator_Count,Maintenance_Candidate
18,V19,TVS,Ntorq,0.120131,8.685968,2.047166,3,1
1,V02,TVS,Raider,0.150929,6.074134,2.047751,2,1
23,V24,Honda,Activa,0.086439,8.788835,1.482013,1,0
22,V23,Suzuki,Access,0.099074,8.411595,1.658542,0,0
24,V25,Suzuki,Access,0.099987,7.385398,1.347186,0,0
14,V15,TVS,Ntorq,0.087341,8.233268,1.149087,0,0
13,V14,TVS,Raider,0.086377,8.141709,1.698995,0,0
11,V12,TVS,Ntorq,0.083982,8.335479,1.573670,0,0
12,V13,Bajaj,Pulsar,0.097709,6.989881,1.211993,0,0
0,V01,Yamaha,Ray ZR,0.081490,8.136740,1.326617,0,0


In [93]:
vehicle_trip_features.to_csv(
    "Preprocessed_Data/vehicle_trip_features.csv",
    index=False
)


In [94]:
vehicle_health.to_csv(
    "Preprocessed_Data/vehicle_health_features.csv",
    index=False
)

In [95]:
print("Number of trip records:",
      len(vehicle_trip_features))

print("Number of vehicles:",
      vehicle_health["Vehicle_ID"].nunique())

print(
    "Vehicles with maintenance candidate flag:",
    vehicle_health[
        "Maintenance_Candidate"
    ].sum()
)


Number of trip records: 450
Number of vehicles: 30
Vehicles with maintenance candidate flag: 2


In [96]:
print(
    vehicle_health[
        [
            "Vehicle_ID",
            "Avg_Accel_Variability",
            "Avg_Gyro_Variability",
            "Avg_Total_Sensor_Spikes",
            "Anomaly_Indicator_Count",
            "Maintenance_Candidate"
        ]
    ]
    .sort_values(
        "Anomaly_Indicator_Count",
        ascending=False
    )
    .to_string(index=False)
)


Vehicle_ID  Avg_Accel_Variability  Avg_Gyro_Variability  Avg_Total_Sensor_Spikes  Anomaly_Indicator_Count  Maintenance_Candidate
       V19               0.120131              8.685968                 2.047166                        3                      1
       V02               0.150929              6.074134                 2.047751                        2                      1
       V24               0.086439              8.788835                 1.482013                        1                      0
       V04               0.051336              7.761656                 0.873708                        0                      0
       V03               0.066163              8.524780                 1.611844                        0                      0
       V01               0.081490              8.136740                 1.326617                        0                      0
       V07               0.047657              6.883234                 0.614842                 

In [97]:
import pandas as pd
import numpy as np

# Load driver-level features created in Step 3
driver_behaviour = pd.read_csv(
    "Preprocessed_Data/driver_behaviour_features.csv"
)

print("Shape:", driver_behaviour.shape)

display(driver_behaviour.head())

Shape: (30, 18)


,Driver_ID,Total_Trips,Total_Harsh_Acceleration,Total_Harsh_Braking,Total_Sudden_Turns,Total_High_Speed_Events,Avg_Risk_Events_Per_10_Min,Avg_Speed,Max_Speed,Avg_Acceleration,Max_Acceleration,Avg_Gyro,Max_Gyro,Driver_Name,Age,Gender,License_Experience_Years,Home_Hub
0,D01,15,11,14,31,19,1.789044,25.381572,54.4,1.026368,1.639733,5.231414,54.095156,Meena Iyer,36,Male,7,Rajajinagar
1,D02,15,15,20,18,13,1.837993,22.676931,50.9,1.049962,1.813246,5.097916,52.919581,Senthil Nair,36,Male,2,HSR Layout
2,D03,15,44,44,30,45,3.370873,22.852751,68.8,1.033505,1.476075,5.736858,54.574108,Vignesh Murugan,35,Male,1,Bommanahalli
3,D04,15,13,15,27,22,1.634954,24.292656,51.6,1.013047,1.561136,5.287398,54.671394,Ganesh Kannan,30,Female,1,HSR Layout
4,D05,15,10,7,12,2,0.565998,23.017252,47.8,1.011880,1.662806,3.934284,45.815206,Manoj Kumar,41,Male,4,Rajajinagar


In [98]:
print(driver_behaviour.columns.tolist())

['Driver_ID', 'Total_Trips', 'Total_Harsh_Acceleration', 'Total_Harsh_Braking', 'Total_Sudden_Turns', 'Total_High_Speed_Events', 'Avg_Risk_Events_Per_10_Min', 'Avg_Speed', 'Max_Speed', 'Avg_Acceleration', 'Max_Acceleration', 'Avg_Gyro', 'Max_Gyro', 'Driver_Name', 'Age', 'Gender', 'License_Experience_Years', 'Home_Hub']


In [99]:
display(
    driver_behaviour[
        [
            "Driver_ID",
            "Driver_Name",
            "Total_Trips",
            "Total_Harsh_Acceleration",
            "Total_Harsh_Braking",
            "Total_Sudden_Turns",
            "Total_High_Speed_Events",
            "Avg_Risk_Events_Per_10_Min"
        ]
    ]
    .sort_values(
        "Driver_ID"
    )
)

,Driver_ID,Driver_Name,Total_Trips,Total_Harsh_Acceleration,Total_Harsh_Braking,Total_Sudden_Turns,Total_High_Speed_Events,Avg_Risk_Events_Per_10_Min
0,D01,Meena Iyer,15,11,14,31,19,1.789044
1,D02,Senthil Nair,15,15,20,18,13,1.837993
2,D03,Vignesh Murugan,15,44,44,30,45,3.370873
3,D04,Ganesh Kannan,15,13,15,27,22,1.634954
4,D05,Manoj Kumar,15,10,7,12,2,0.565998
5,D06,Bhavani Raj,15,57,48,35,73,4.461658
6,D07,Kavya Pillai,15,14,19,26,24,1.870861
7,D08,Arun Raj,15,16,14,22,16,1.402142
8,D09,Rajesh Raj,15,13,9,3,0,0.583109
9,D10,Mohan Pillai,15,13,14,24,13,1.310201


In [100]:
#Harsh acceleration
driver_behaviour["Harsh_Acceleration_Risk"] = (
    driver_behaviour[
        "Total_Harsh_Acceleration"
    ].rank(pct=True) * 100
)

#Harsh braking
driver_behaviour["Harsh_Braking_Risk"] = (
    driver_behaviour[
        "Total_Harsh_Braking"
    ].rank(pct=True) * 100
)

#Sudden turns
driver_behaviour["Sudden_Turn_Risk"] = (
    driver_behaviour[
        "Total_Sudden_Turns"
    ].rank(pct=True) * 100
)

#High-speed events

driver_behaviour["High_Speed_Risk"] = (
    driver_behaviour[
        "Total_High_Speed_Events"
    ].rank(pct=True) * 100
)

In [101]:
driver_behaviour["Risk_Frequency_Risk"] = (
    driver_behaviour[
        "Avg_Risk_Events_Per_10_Min"
    ].rank(pct=True) * 100
)

In [102]:
display(
    driver_behaviour[
        [
            "Driver_ID",
            "Driver_Name",
            "Harsh_Acceleration_Risk",
            "Harsh_Braking_Risk",
            "Sudden_Turn_Risk",
            "High_Speed_Risk",
            "Risk_Frequency_Risk"
        ]
    ]
)

,Driver_ID,Driver_Name,Harsh_Acceleration_Risk,Harsh_Braking_Risk,Sudden_Turn_Risk,High_Speed_Risk,Risk_Frequency_Risk
0,D01,Meena Iyer,21.666667,43.333333,90.000000,56.666667,66.666667
1,D02,Senthil Nair,48.333333,71.666667,40.000000,45.000000,70.000000
2,D03,Vignesh Murugan,86.666667,86.666667,86.666667,83.333333,80.000000
3,D04,Ganesh Kannan,33.333333,53.333333,71.666667,70.000000,53.333333
4,D05,Manoj Kumar,16.666667,8.333333,21.666667,25.000000,10.000000
5,D06,Bhavani Raj,100.000000,90.000000,96.666667,96.666667,93.333333
6,D07,Kavya Pillai,40.000000,65.000000,63.333333,76.666667,73.333333
7,D08,Arun Raj,60.000000,43.333333,50.000000,50.000000,36.666667
8,D09,Rajesh Raj,33.333333,20.000000,3.333333,8.333333,13.333333
9,D10,Mohan Pillai,33.333333,43.333333,60.000000,45.000000,33.333333


In [103]:
driver_behaviour["Driver_Risk_Score"] = (
    0.30 * driver_behaviour["Harsh_Braking_Risk"]
    +
    0.25 * driver_behaviour["Harsh_Acceleration_Risk"]
    +
    0.20 * driver_behaviour["Sudden_Turn_Risk"]
    +
    0.15 * driver_behaviour["High_Speed_Risk"]
    +
    0.10 * driver_behaviour["Risk_Frequency_Risk"]
)

In [104]:
print(
    "Minimum:",
    driver_behaviour["Driver_Risk_Score"].min()
)

print(
    "Maximum:",
    driver_behaviour["Driver_Risk_Score"].max()
)

print(
    "Mean:",
    driver_behaviour["Driver_Risk_Score"].mean()
)

Minimum: 6.166666666666667
Maximum: 98.41666666666667
Mean: 51.66666666666666


In [105]:
driver_behaviour["Risk_Category"] = pd.cut(
    driver_behaviour["Driver_Risk_Score"],
    bins=[-np.inf, 33.33, 66.67, np.inf],
    labels=[
        "Low Risk",
        "Moderate Risk",
        "High Risk"
    ]
)

In [106]:
print(
    driver_behaviour[
        "Risk_Category"
    ].value_counts()
)

Risk_Category
Moderate Risk    14
Low Risk          8
High Risk         8
Name: count, dtype: int64


In [107]:
driver_behaviour["Total_Risk_Events"] = (
    driver_behaviour[
        "Total_Harsh_Acceleration"
    ]
    +
    driver_behaviour[
        "Total_Harsh_Braking"
    ]
    +
    driver_behaviour[
        "Total_Sudden_Turns"
    ]
    +
    driver_behaviour[
        "Total_High_Speed_Events"
    ]
)

In [108]:
driver_behaviour["Risk_Events_Per_Trip"] = (
    driver_behaviour["Total_Risk_Events"]
    /
    driver_behaviour["Total_Trips"]
)
#Round it:

driver_behaviour["Risk_Events_Per_Trip"] = (
    driver_behaviour["Risk_Events_Per_Trip"]
    .round(2)
)

In [109]:
driver_behaviour["Risk_Rank"] = (
    driver_behaviour[
        "Driver_Risk_Score"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

In [110]:
display(
    driver_behaviour[
        [
            "Risk_Rank",
            "Driver_ID",
            "Driver_Name",
            "Driver_Risk_Score",
            "Risk_Category",
            "Total_Risk_Events",
            "Risk_Events_Per_Trip"
        ]
    ]
    .sort_values(
        "Risk_Rank"
    )
)


,Risk_Rank,Driver_ID,Driver_Name,Driver_Risk_Score,Risk_Category,Total_Risk_Events,Risk_Events_Per_Trip
23,1,D24,Lakshmi Iyer,98.416667,High Risk,220,14.67
5,2,D06,Bhavani Raj,95.166667,High Risk,213,14.20
22,3,D23,Kavya Pillai,93.083333,High Risk,199,13.27
13,4,D14,Rajesh Subramaniam,91.000000,High Risk,188,12.53
2,5,D03,Vignesh Murugan,85.500000,High Risk,163,10.87
18,6,D19,Senthil Pillai,82.166667,High Risk,155,10.33
11,7,D12,Praveen Murugan,81.000000,High Risk,119,7.93
14,8,D15,Meena Velu,69.000000,High Risk,87,5.80
20,9,D21,Vignesh Raj,63.833333,Moderate Risk,86,5.73
6,10,D07,Kavya Pillai,61.000000,Moderate Risk,83,5.53


In [111]:
risk_columns = [
    "Harsh_Acceleration_Risk",
    "Harsh_Braking_Risk",
    "Sudden_Turn_Risk",
    "High_Speed_Risk",
    "Risk_Frequency_Risk"
]

In [112]:
driver_behaviour["Primary_Risk_Factor"] = (
    driver_behaviour[
        risk_columns
    ]
    .idxmax(axis=1)
)

In [113]:
risk_factor_map = {
    "Harsh_Acceleration_Risk":
        "Harsh acceleration",

    "Harsh_Braking_Risk":
        "Harsh braking",

    "Sudden_Turn_Risk":
        "Sudden turning",

    "High_Speed_Risk":
        "High-speed events",

    "Risk_Frequency_Risk":
        "High risk-event frequency"
}


In [114]:
driver_behaviour["Primary_Risk_Factor"] = (
    driver_behaviour[
        "Primary_Risk_Factor"
    ].map(risk_factor_map)
)

In [115]:
driver_behaviour["Braking_Contribution"] = (
    0.30 *
    driver_behaviour["Harsh_Braking_Risk"]
)

driver_behaviour["Acceleration_Contribution"] = (
    0.25 *
    driver_behaviour["Harsh_Acceleration_Risk"]
)

driver_behaviour["Turning_Contribution"] = (
    0.20 *
    driver_behaviour["Sudden_Turn_Risk"]
)

driver_behaviour["Speed_Contribution"] = (
    0.15 *
    driver_behaviour["High_Speed_Risk"]
)

driver_behaviour["Frequency_Contribution"] = (
    0.10 *
    driver_behaviour["Risk_Frequency_Risk"]
)


In [116]:
driver_behaviour["Score_Check"] = (
    driver_behaviour["Braking_Contribution"]
    +
    driver_behaviour["Acceleration_Contribution"]
    +
    driver_behaviour["Turning_Contribution"]
    +
    driver_behaviour["Speed_Contribution"]
    +
    driver_behaviour["Frequency_Contribution"]
)

In [117]:
np.allclose(
    driver_behaviour["Driver_Risk_Score"],
    driver_behaviour["Score_Check"]
)

True

In [118]:
driver_behaviour["Risk_Explanation"] = (
    driver_behaviour["Primary_Risk_Factor"]
    + " is the strongest contributor to the "
    + "driver's relative risk profile."
)

In [119]:
driver_ranking = driver_behaviour.sort_values(
    "Driver_Risk_Score",
    ascending=False
)
#Display the most important columns:

display(
    driver_ranking[
        [
            "Risk_Rank",
            "Driver_ID",
            "Driver_Name",
            "Driver_Risk_Score",
            "Risk_Category",
            "Total_Risk_Events",
            "Risk_Events_Per_Trip",
            "Primary_Risk_Factor"
        ]
    ]
)

,Risk_Rank,Driver_ID,Driver_Name,Driver_Risk_Score,Risk_Category,Total_Risk_Events,Risk_Events_Per_Trip,Primary_Risk_Factor
23,1,D24,Lakshmi Iyer,98.416667,High Risk,220,14.67,Harsh braking
5,2,D06,Bhavani Raj,95.166667,High Risk,213,14.20,Harsh acceleration
22,3,D23,Kavya Pillai,93.083333,High Risk,199,13.27,Harsh acceleration
13,4,D14,Rajesh Subramaniam,91.000000,High Risk,188,12.53,High risk-event frequency
2,5,D03,Vignesh Murugan,85.500000,High Risk,163,10.87,Harsh acceleration
18,6,D19,Senthil Pillai,82.166667,High Risk,155,10.33,High risk-event frequency
11,7,D12,Praveen Murugan,81.000000,High Risk,119,7.93,Sudden turning
14,8,D15,Meena Velu,69.000000,High Risk,87,5.80,High risk-event frequency
20,9,D21,Vignesh Raj,63.833333,Moderate Risk,86,5.73,Harsh acceleration
6,10,D07,Kavya Pillai,61.000000,Moderate Risk,83,5.53,High-speed events


In [120]:
print(
    driver_behaviour[
        "Driver_Risk_Score"
    ].describe()
)


print(
    driver_behaviour[
        "Driver_Risk_Score"
    ].quantile(
        [0.25, 0.50, 0.75]
    )
)


print(
    driver_behaviour[
        "Risk_Category"
    ].value_counts()
)

count    30.000000
mean     51.666667
std      27.413561
min       6.166667
25%      31.812500
50%      53.041667
75%      67.708333
max      98.416667
Name: Driver_Risk_Score, dtype: float64
0.25    31.812500
0.50    53.041667
0.75    67.708333
Name: Driver_Risk_Score, dtype: float64
Risk_Category
Moderate Risk    14
Low Risk          8
High Risk         8
Name: count, dtype: int64


In [122]:
driver_behaviour.to_csv(
    "Preprocessed_Data/driver_risk_scores.csv",
    index=False
)

In [123]:
print(
    driver_behaviour[
        [
            "Risk_Rank",
            "Driver_ID",
            "Driver_Name",
            "Driver_Risk_Score",
            "Risk_Category",
            "Total_Risk_Events",
            "Risk_Events_Per_Trip",
            "Primary_Risk_Factor"
        ]
    ]
    .sort_values("Risk_Rank")
    .to_string(index=False)
)


print(
    driver_behaviour[
        "Risk_Category"
    ].value_counts()
)


print(
    driver_behaviour[
        "Driver_Risk_Score"
    ].describe()
)

 Risk_Rank Driver_ID        Driver_Name  Driver_Risk_Score Risk_Category  Total_Risk_Events  Risk_Events_Per_Trip       Primary_Risk_Factor
         1       D24       Lakshmi Iyer          98.416667     High Risk                220                 14.67             Harsh braking
         2       D06        Bhavani Raj          95.166667     High Risk                213                 14.20        Harsh acceleration
         3       D23       Kavya Pillai          93.083333     High Risk                199                 13.27        Harsh acceleration
         4       D14 Rajesh Subramaniam          91.000000     High Risk                188                 12.53 High risk-event frequency
         5       D03    Vignesh Murugan          85.500000     High Risk                163                 10.87        Harsh acceleration
         6       D19     Senthil Pillai          82.166667     High Risk                155                 10.33 High risk-event frequency
         7       D12

In [125]:
import pandas as pd
import numpy as np

vehicle_health = pd.read_csv(
    "Preprocessed_Data/vehicle_health_features.csv"
)

print(vehicle_health.shape)
display(vehicle_health.head())

(30, 29)


,Vehicle_ID,Total_Trips,Avg_Accel_Variability,Max_Accel_Variability,Avg_Accel_Range,Avg_Gyro_Variability,Max_Gyro_Variability,Avg_Gyro_Range,Avg_Accel_Spikes_Per_10_Min,Avg_Gyro_Spikes_Per_10_Min,...,Vehicle_Age_Years,Days_Since_Last_Service,Accel_Variability_Percentile,Gyro_Variability_Percentile,Sensor_Spike_Percentile,High_Accel_Variability,High_Gyro_Variability,High_Sensor_Spikes,Anomaly_Indicator_Count,Maintenance_Candidate
0,V01,15,0.081490,0.180949,0.376312,8.136740,12.981530,38.011426,0.734773,0.591845,...,5,74,0.700000,0.700000,0.733333,0,0,0,0,0
1,V02,16,0.150929,0.255354,0.663612,6.074134,13.277175,27.148729,1.156757,0.890994,...,5,107,1.000000,0.433333,1.000000,1,0,1,2,1
2,V03,16,0.066163,0.105605,0.291966,8.524780,13.232653,40.321897,0.917712,0.694132,...,4,44,0.433333,0.900000,0.866667,0,0,0,0,0
3,V04,15,0.051336,0.130241,0.248070,7.761656,11.435348,35.766309,0.246294,0.627415,...,6,33,0.133333,0.666667,0.533333,0,0,0,0,0
4,V05,15,0.061867,0.143864,0.304280,4.584335,8.466150,25.626137,0.176754,0.294238,...,1,40,0.333333,0.300000,0.166667,0,0,0,0,0


In [126]:
metrics = [
    "Avg_Accel_Variability",
    "Avg_Gyro_Variability",
    "Avg_Total_Sensor_Spikes"
]

display(
    vehicle_health[
        ["Vehicle_ID"] + metrics
    ].sort_values(
        "Vehicle_ID"
    )
)

,Vehicle_ID,Avg_Accel_Variability,Avg_Gyro_Variability,Avg_Total_Sensor_Spikes
0,V01,0.081490,8.136740,1.326617
1,V02,0.150929,6.074134,2.047751
2,V03,0.066163,8.524780,1.611844
3,V04,0.051336,7.761656,0.873708
4,V05,0.061867,4.584335,0.470992
5,V06,0.063971,8.681526,1.266847
6,V07,0.047657,6.883234,0.614842
7,V08,0.067146,5.791958,0.852581
8,V09,0.052504,3.244205,0.308394
9,V10,0.081066,5.701535,0.812802


In [127]:


vehicle_health["Accel_Percentile"] = (
    vehicle_health["Avg_Accel_Variability"]
    .rank(pct=True)
)

vehicle_health["Gyro_Percentile"] = (
    vehicle_health["Avg_Gyro_Variability"]
    .rank(pct=True)
)



vehicle_health["Spike_Percentile"] = (
    vehicle_health["Avg_Total_Sensor_Spikes"]
    .rank(pct=True)
)

In [128]:
display(
    vehicle_health[
        [
            "Vehicle_ID",
            "Accel_Percentile",
            "Gyro_Percentile",
            "Spike_Percentile"
        ]
    ].sort_values(
        "Accel_Percentile",
        ascending=False
    )
)

,Vehicle_ID,Accel_Percentile,Gyro_Percentile,Spike_Percentile
1,V02,1.000000,0.433333,1.000000
18,V19,0.966667,0.966667,0.966667
24,V25,0.933333,0.633333,0.766667
22,V23,0.900000,0.833333,0.900000
12,V13,0.866667,0.600000,0.633333
14,V15,0.833333,0.766667,0.600000
23,V24,0.800000,1.000000,0.800000
13,V14,0.766667,0.733333,0.933333
11,V12,0.733333,0.800000,0.833333
0,V01,0.700000,0.700000,0.733333


In [129]:
vehicle_health["Accel_Anomaly_Risk"] = (
    vehicle_health["Accel_Percentile"] * 100
)

vehicle_health["Gyro_Anomaly_Risk"] = (
    vehicle_health["Gyro_Percentile"] * 100
)

vehicle_health["Spike_Anomaly_Risk"] = (
    vehicle_health["Spike_Percentile"] * 100
)


In [130]:
vehicle_health["Accel_Health"] = (
    100 - vehicle_health["Accel_Anomaly_Risk"]
)

vehicle_health["Gyro_Health"] = (
    100 - vehicle_health["Gyro_Anomaly_Risk"]
)

vehicle_health["Spike_Health"] = (
    100 - vehicle_health["Spike_Anomaly_Risk"]
)


In [131]:
vehicle_health["Vehicle_Health_Score"] = (
    0.40 * vehicle_health["Accel_Health"]
    +
    0.35 * vehicle_health["Gyro_Health"]
    +
    0.25 * vehicle_health["Spike_Health"]
)


In [132]:
print(
    "Minimum:",
    vehicle_health["Vehicle_Health_Score"].min()
)

print(
    "Maximum:",
    vehicle_health["Vehicle_Health_Score"].max()
)

print(
    "Mean:",
    vehicle_health["Vehicle_Health_Score"].mean()
)


Minimum: 3.3333333333333286
Maximum: 92.83333333333333
Mean: 48.33333333333334


In [133]:
assert vehicle_health[
    "Vehicle_Health_Score"
].between(0, 100).all()

In [134]:
vehicle_health["Health_Category"] = pd.cut(
    vehicle_health["Vehicle_Health_Score"],
    bins=[-np.inf, 49.99, 74.99, np.inf],
    labels=[
        "Maintenance Review",
        "Monitor",
        "Healthy"
    ]
)

In [135]:
vehicle_health["Maintenance_Priority"] = np.select(
    [
        vehicle_health["Vehicle_Health_Score"] < 50,
        vehicle_health["Vehicle_Health_Score"] < 75
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)



In [136]:
vehicle_health["Primary_Health_Concern"] = (
    vehicle_health[
        [
            "Accel_Anomaly_Risk",
            "Gyro_Anomaly_Risk",
            "Spike_Anomaly_Risk"
        ]
    ].idxmax(axis=1)
)

#Map the technical names:

concern_map = {
    "Accel_Anomaly_Risk":
        "High acceleration variability",

    "Gyro_Anomaly_Risk":
        "High gyroscope variability",

    "Spike_Anomaly_Risk":
        "High sensor spike frequency"
}

vehicle_health["Primary_Health_Concern"] = (
    vehicle_health["Primary_Health_Concern"]
    .map(concern_map)
)


In [137]:
vehicle_health["Acceleration_Contribution"] = (
    0.40 * vehicle_health["Accel_Health"]
)

vehicle_health["Gyroscope_Contribution"] = (
    0.35 * vehicle_health["Gyro_Health"]
)

vehicle_health["Spike_Contribution"] = (
    0.25 * vehicle_health["Spike_Health"]
)


In [138]:
vehicle_health["Score_Check"] = (
    vehicle_health["Acceleration_Contribution"]
    +
    vehicle_health["Gyroscope_Contribution"]
    +
    vehicle_health["Spike_Contribution"]
)


In [139]:
print(
    np.allclose(
        vehicle_health["Vehicle_Health_Score"],
        vehicle_health["Score_Check"]
    )
)


True


In [140]:
vehicle_health["Maintenance_Rank"] = (
    vehicle_health["Vehicle_Health_Score"]
    .rank(
        ascending=True,
        method="min"
    )
    .astype(int)
)


In [141]:
vehicle_health["Health_Explanation"] = (
    vehicle_health["Primary_Health_Concern"]
    + " is the strongest contributor to the "
    + "vehicle's relative sensor-anomaly profile."
)

In [142]:
display(
    vehicle_health[
        vehicle_health["Vehicle_ID"].isin(
            ["V19", "V02", "V24"]
        )
    ][
        [
            "Vehicle_ID",
            "Avg_Accel_Variability",
            "Avg_Gyro_Variability",
            "Avg_Total_Sensor_Spikes",
            "Accel_Anomaly_Risk",
            "Gyro_Anomaly_Risk",
            "Spike_Anomaly_Risk",
            "Vehicle_Health_Score",
            "Health_Category",
            "Maintenance_Priority",
            "Primary_Health_Concern"
        ]
    ]
)

,Vehicle_ID,Avg_Accel_Variability,Avg_Gyro_Variability,Avg_Total_Sensor_Spikes,Accel_Anomaly_Risk,Gyro_Anomaly_Risk,Spike_Anomaly_Risk,Vehicle_Health_Score,Health_Category,Maintenance_Priority,Primary_Health_Concern
1,V02,0.150929,6.074134,2.047751,100.000000,43.333333,100.000000,19.833333,Maintenance Review,High,High acceleration variability
18,V19,0.120131,8.685968,2.047166,96.666667,96.666667,96.666667,3.333333,Maintenance Review,High,High acceleration variability
23,V24,0.086439,8.788835,1.482013,80.000000,100.000000,80.000000,13.000000,Maintenance Review,High,High gyroscope variability


In [143]:
display(
    vehicle_health[
        [
            "Maintenance_Rank",
            "Vehicle_ID",
            "Vehicle_Health_Score",
            "Health_Category",
            "Maintenance_Priority",
            "Primary_Health_Concern"
        ]
    ].sort_values(
        "Maintenance_Rank"
    )
)

,Maintenance_Rank,Vehicle_ID,Vehicle_Health_Score,Health_Category,Maintenance_Priority,Primary_Health_Concern
18,1,V19,3.333333,Maintenance Review,High,High acceleration variability
22,2,V23,12.333333,Maintenance Review,High,High acceleration variability
23,3,V24,13.000000,Maintenance Review,High,High gyroscope variability
1,4,V02,19.833333,Maintenance Review,High,High acceleration variability
13,5,V14,20.333333,Maintenance Review,High,High sensor spike frequency
24,6,V25,21.333333,Maintenance Review,High,High acceleration variability
11,7,V12,21.833333,Maintenance Review,High,High sensor spike frequency
14,8,V15,24.833333,Maintenance Review,High,High acceleration variability
12,9,V13,28.500000,Maintenance Review,High,High acceleration variability
0,10,V01,29.166667,Maintenance Review,High,High sensor spike frequency


In [144]:
print(
    vehicle_health[
        "Health_Category"
    ].value_counts()
)

#And:

print(
    vehicle_health[
        "Maintenance_Priority"
    ].value_counts()
)

Health_Category
Maintenance Review    15
Monitor               10
Healthy                5
Name: count, dtype: int64
Maintenance_Priority
High      15
Medium    10
Low        5
Name: count, dtype: int64


In [145]:
vehicle_health.to_csv(
    "Preprocessed_Data/vehicle_health_scores.csv",
    index=False
)


In [146]:
print(
    vehicle_health[
        [
            "Vehicle_ID",
            "Vehicle_Health_Score",
            "Health_Category",
            "Maintenance_Priority",
            "Primary_Health_Concern"
        ]
    ]
    .sort_values(
        "Vehicle_Health_Score"
    )
    .to_string(index=False)
)


Vehicle_ID  Vehicle_Health_Score    Health_Category Maintenance_Priority        Primary_Health_Concern
       V19              3.333333 Maintenance Review                 High High acceleration variability
       V23             12.333333 Maintenance Review                 High High acceleration variability
       V24             13.000000 Maintenance Review                 High    High gyroscope variability
       V02             19.833333 Maintenance Review                 High High acceleration variability
       V14             20.333333 Maintenance Review                 High   High sensor spike frequency
       V25             21.333333 Maintenance Review                 High High acceleration variability
       V12             21.833333 Maintenance Review                 High   High sensor spike frequency
       V15             24.833333 Maintenance Review                 High High acceleration variability
       V13             28.500000 Maintenance Review                 High 

In [148]:
print(
    vehicle_health[
        "Health_Category"
    ].value_counts()
)


Health_Category
Maintenance Review    15
Monitor               10
Healthy                5
Name: count, dtype: int64


In [149]:
print(
    vehicle_health[
        "Vehicle_Health_Score"
    ].describe()
)

count    30.000000
mean     48.333333
std      25.853922
min       3.333333
25%      25.750000
50%      50.916667
75%      70.666667
max      92.833333
Name: Vehicle_Health_Score, dtype: float64


In [150]:
display(
    vehicle_health[
        vehicle_health["Vehicle_ID"].isin(
            ["V19", "V02", "V24"]
        )
    ][
        [
            "Vehicle_ID",
            "Avg_Accel_Variability",
            "Avg_Gyro_Variability",
            "Avg_Total_Sensor_Spikes",
            "Vehicle_Health_Score",
            "Health_Category",
            "Maintenance_Priority",
            "Primary_Health_Concern"
        ]
    ]
)

,Vehicle_ID,Avg_Accel_Variability,Avg_Gyro_Variability,Avg_Total_Sensor_Spikes,Vehicle_Health_Score,Health_Category,Maintenance_Priority,Primary_Health_Concern
1,V02,0.150929,6.074134,2.047751,19.833333,Maintenance Review,High,High acceleration variability
18,V19,0.120131,8.685968,2.047166,3.333333,Maintenance Review,High,High acceleration variability
23,V24,0.086439,8.788835,1.482013,13.000000,Maintenance Review,High,High gyroscope variability
